# Visual Inspection — Selected 10 Planets (5 Hot Jupiters + 5 Hot Saturns)

Downloads and plots the TESS PDC-SAP light curve for each of the 10 selected planets in their chosen sector. Transit times are marked for visual verification. Each planet uses its own best sector (not necessarily all the same sector).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import lightkurve as lk
import warnings
warnings.filterwarnings("ignore")

# ── Load selected planets ─────────────────────────────────────────────────────
df = pd.read_csv("../selected_10_planets.csv")

# Each planet now has its own sector column
print("Selected planets:")
print(df[["System", "type", "R_b", "Period", "depth_pct", "T14_hr", "sector"]].to_string(index=False))


In [2]:
def download_lc(host_star, sector, retries=3):
    """Download TESS SPOC 2-min PDC-SAP light curve, normalized to median=1."""
    for attempt in range(retries):
        try:
            sr = lk.search_lightcurve(host_star, mission="TESS", sector=sector,
                                       author="SPOC", exptime=120)
            if sr is None or len(sr) == 0:
                return None
            lc = sr[0].download(flux_column="pdcsap_flux")
            if lc is None:
                return None
            lc = lc.remove_nans()
            # Normalize to median = 1
            med = np.nanmedian(lc.flux.value)
            lc = lk.LightCurve(time=lc.time, flux=lc.flux.value / med)
            return lc
        except Exception as e:
            if attempt < retries - 1:
                import time; time.sleep(10)
            else:
                print(f"    FAILED after {retries} tries: {e}")
    return None


def get_transit_centers(t_arr, t0, period, t14_hr):
    """Predict transit mid-times that fall within the time array."""
    half_dur = (t14_hr / 24) / 2
    n_lo = int(np.ceil((t_arr.min() - t0) / period))
    n_hi = int(np.floor((t_arr.max() - t0) / period))
    tcs = []
    for n in range(n_lo, n_hi + 1):
        tc = t0 + n * period
        if t_arr.min() - half_dur <= tc <= t_arr.max() + half_dur:
            tcs.append(tc)
    return tcs


def estimate_t0(lc, period):
    """Rough t0 estimate: fold the LC and find the minimum (transit dip)."""
    t = lc.time.value
    f = lc.flux.value
    # Scan t0 candidates at 0.01*period resolution
    best_t0, best_depth = t[0], 0
    for t0_try in np.linspace(t[0], t[0] + period, 200):
        phase = (t - t0_try) % period
        phase[phase > period / 2] -= period
        in_transit = np.abs(phase) < 0.05 * period
        if in_transit.sum() < 3:
            continue
        depth = 1.0 - np.median(f[in_transit])
        if depth > best_depth:
            best_depth = depth
            best_t0 = t0_try
    return best_t0


print("Helper functions defined. Ready to download light curves.")


Helper functions defined. Ready to download light curves.


In [ ]:
import time as time_mod

# Download — each planet uses its own sector from the CSV
lc_cache = {}

for _, row in df.iterrows():
    planet = row["System"]
    host   = row["host_star"]
    sector = int(row["sector"])
    print(f"Downloading {planet} ({row['type']}) S{sector}...", end=" ", flush=True)
    lc = download_lc(host, sector)
    if lc is not None:
        lc_cache[planet] = (lc, sector)
        print(f"{len(lc.flux.value)} points")
    else:
        print("NO DATA")
    time_mod.sleep(0.5)

print(f"\nDownloaded {len(lc_cache)}/10 light curves successfully.")


## Full-Sector Light Curves

Each panel shows the complete Sector 74 PDC-SAP light curve. Red dashed vertical lines mark predicted transit mid-times based on the TEPCat period (T0 estimated from the light curve itself). The transit depth (%) and duration (hr) are shown in the title.

In [ ]:
RJUP_RSUN = 0.10045
RSUN_AU   = 0.00465047

tepcat_full = pd.read_csv("../data/planets_ready_for_modeling.csv")
for col in ["R_b", "R_A", "a(AU)", "Period"]:
    tepcat_full[col] = pd.to_numeric(tepcat_full[col], errors="coerce")
tepcat_full["k"]    = tepcat_full["R_b"] * RJUP_RSUN / tepcat_full["R_A"]
tepcat_full["aRs"]  = tepcat_full["a(AU)"] / (tepcat_full["R_A"] * RSUN_AU)
tepcat_full["depth_pct"] = (tepcat_full["k"] ** 2) * 100
arg = ((1 + tepcat_full["k"]) / tepcat_full["aRs"]).clip(-1, 1)
tepcat_full["T14_hr"] = (tepcat_full["Period"] * 24 / np.pi) * np.arcsin(arg)
params = tepcat_full.set_index("System")[["depth_pct", "T14_hr"]].to_dict("index")

type_colors = {"Hot Jupiter": "#e8534a", "Hot Saturn": "#e8a23a"}

hj_planets = df[df["type"] == "Hot Jupiter"]["System"].tolist()
hs_planets = df[df["type"] == "Hot Saturn"]["System"].tolist()
all_planets = hj_planets + hs_planets

n = len(all_planets)
fig, axes = plt.subplots(n, 2, figsize=(18, n * 3.5))
fig.suptitle("Selected 10 Planets — Hot Jupiters (red) & Hot Saturns (orange)\n"
             "Left: Full LC   Right: Phase-folded",
             fontsize=14, fontweight="bold", y=1.01)

for i, planet in enumerate(all_planets):
    row_data = df[df["System"] == planet].iloc[0]
    ptype    = row_data["type"]
    period   = float(row_data["Period"])
    rb       = float(row_data["R_b"])
    teff     = int(row_data["Teff"])
    sector   = int(row_data["sector"])
    depth    = params.get(planet, {}).get("depth_pct", float("nan"))
    t14      = params.get(planet, {}).get("T14_hr",    float("nan"))
    color    = type_colors[ptype]

    ax_lc   = axes[i, 0]
    ax_fold = axes[i, 1]

    if planet not in lc_cache:
        for ax in (ax_lc, ax_fold):
            ax.text(0.5, 0.5, "No data", transform=ax.transAxes,
                    ha="center", va="center", fontsize=12, color="gray")
            ax.set_title(f"{planet} [{ptype}]")
        continue

    lc, sec = lc_cache[planet]
    tarr = lc.time.value
    farr = lc.flux.value

    t0  = estimate_t0(lc, period)
    tcs = get_transit_centers(tarr, t0, period, t14 if np.isfinite(t14) else period * 0.05)

    # Full LC
    ax_lc.plot(tarr, farr, ".", color=color, ms=1.2, alpha=0.5, rasterized=True)
    for tc in tcs:
        ax_lc.axvline(tc, color="red", lw=0.8, alpha=0.7, ls="--")
    ax_lc.set_xlabel("BTJD (days)", fontsize=9)
    ax_lc.set_ylabel("Norm. flux", fontsize=9)
    ax_lc.set_title(
        f"{planet}   [{ptype}]   R_b={rb:.2f} Rjup   P={period:.3f}d   "
        f"Teff={teff}K   S{sec}\n"
        f"depth≈{depth:.3f}%   T14≈{t14:.2f}hr   {len(tcs)} transits marked",
        fontsize=8.5, loc="left"
    )
    ax_lc.tick_params(labelsize=8)

    # Phase-folded
    phase = (tarr - t0) % period
    phase[phase > period / 2] -= period
    half_show = min(3 * (t14 / 24), period / 2) if np.isfinite(t14) else period / 2
    mask_fold = np.abs(phase) <= half_show

    ax_fold.plot(phase[mask_fold] * 24, farr[mask_fold],
                 ".", color=color, ms=1.8, alpha=0.6, rasterized=True)
    ax_fold.axvline(0, color="red", lw=1.0, ls="--", alpha=0.8, label="mid-transit")
    if np.isfinite(t14):
        ax_fold.axvspan(-t14 / 2, t14 / 2, alpha=0.08, color="red",
                        label=f"T14={t14:.2f}hr")
    ax_fold.set_xlabel("Phase (hours from mid-transit)", fontsize=9)
    ax_fold.set_ylabel("Norm. flux", fontsize=9)
    ax_fold.set_title(f"Phase-folded (P={period:.3f}d)", fontsize=8.5, loc="left")
    ax_fold.legend(fontsize=7, loc="lower right")
    ax_fold.tick_params(labelsize=8)

    for ax in (ax_lc, ax_fold):
        ax.spines["left"].set_color(color)
        ax.spines["left"].set_linewidth(2.5)

plt.tight_layout()
plt.savefig("../results/selected_planets_HJ_HS.png", dpi=130, bbox_inches="tight")
plt.show()
print("Saved: results/selected_planets_HJ_HS.png")


## Individual Zoom-In Plots

One page per planet — shows the full LC plus a zoom into a single representative transit window, so you can see ingress/egress clearly.

In [ ]:
for planet in all_planets:
    if planet not in lc_cache:
        print(f"Skipping {planet} — no data")
        continue

    row_data = df[df["System"] == planet].iloc[0]
    ptype    = row_data["type"]
    period   = float(row_data["Period"])
    rb       = float(row_data["R_b"])
    sector   = int(row_data["sector"])
    depth    = params.get(planet, {}).get("depth_pct", float("nan"))
    t14      = params.get(planet, {}).get("T14_hr",    float("nan"))
    color    = type_colors[ptype]

    lc, sec = lc_cache[planet]
    tarr = lc.time.value
    farr = lc.flux.value

    t0  = estimate_t0(lc, period)
    tcs = get_transit_centers(tarr, t0, period, t14 if np.isfinite(t14) else period * 0.05)

    fig = plt.figure(figsize=(16, 8))
    gs  = gridspec.GridSpec(2, 2, figure=fig, height_ratios=[1, 1.4],
                            hspace=0.45, wspace=0.3)
    ax_full  = fig.add_subplot(gs[0, :])
    ax_zoom  = fig.add_subplot(gs[1, 0])
    ax_fold  = fig.add_subplot(gs[1, 1])

    # Full LC
    ax_full.plot(tarr, farr, ".", color=color, ms=1.2, alpha=0.4, rasterized=True)
    for tc in tcs:
        ax_full.axvline(tc, color="red", lw=0.7, alpha=0.6, ls="--")
    ax_full.set_xlabel("BTJD (days)", fontsize=10)
    ax_full.set_ylabel("Norm. flux", fontsize=10)
    ax_full.set_title(
        f"{planet}  [{ptype}]  |  R_b={rb:.2f} Rjup  |  P={period:.3f}d  |  "
        f"depth≈{depth:.3f}%  |  T14≈{t14:.2f}hr  |  {len(tcs)} transits  |  S{sec}",
        fontsize=10, fontweight="bold", loc="left"
    )

    # Single transit zoom
    t_mid = (tarr.min() + tarr.max()) / 2
    if tcs:
        tc_best = min(tcs, key=lambda tc: abs(tc - t_mid))
        half_win = max(t14 / 24 * 2.5, period * 0.08) if np.isfinite(t14) else period * 0.08
        mask_z = (tarr >= tc_best - half_win) & (tarr <= tc_best + half_win)
        ax_zoom.plot(tarr[mask_z], farr[mask_z], ".", color=color, ms=3, alpha=0.8)
        ax_zoom.axvline(tc_best, color="red", lw=1, ls="--", alpha=0.9)
        if np.isfinite(t14):
            ax_zoom.axvspan(tc_best - t14 / 48, tc_best + t14 / 48, alpha=0.10, color="red")
        ax_zoom.set_xlabel("BTJD (days)", fontsize=10)
        ax_zoom.set_ylabel("Norm. flux", fontsize=10)
        ax_zoom.set_title("Zoom: single transit", fontsize=10)
    else:
        ax_zoom.text(0.5, 0.5, "No transits in sector", transform=ax_zoom.transAxes,
                     ha="center", va="center", color="gray")

    # Phase-folded
    phase = (tarr - t0) % period
    phase[phase > period / 2] -= period
    half_show = min(3 * t14 / 24, period / 2) if np.isfinite(t14) else period / 2
    mask_f = np.abs(phase) <= half_show
    ax_fold.plot(phase[mask_f] * 24, farr[mask_f],
                 ".", color=color, ms=2, alpha=0.5, rasterized=True)
    ax_fold.axvline(0, color="red", lw=1, ls="--", alpha=0.9)
    if np.isfinite(t14):
        ax_fold.axvspan(-t14 / 2, t14 / 2, alpha=0.08, color="red", label=f"T14={t14:.2f}hr")
        ax_fold.legend(fontsize=8)
    ax_fold.set_xlabel("Hours from mid-transit", fontsize=10)
    ax_fold.set_ylabel("Norm. flux", fontsize=10)
    ax_fold.set_title("Phase-folded", fontsize=10)

    for ax in (ax_full, ax_zoom, ax_fold):
        ax.spines["left"].set_color(color)
        ax.spines["left"].set_linewidth(2)
        ax.tick_params(labelsize=9)

    plt.suptitle(f"S{sec} — {planet}  [{ptype}]", fontsize=13, y=1.01)
    fname = planet.replace(" ", "_").replace("/", "_")
    plt.savefig(f"../results/lc_{fname}_S{sec}.png", dpi=130, bbox_inches="tight")
    plt.show()
    print(f"Saved: results/lc_{fname}_S{sec}.png")
